# S3SessionManager — Cloud Persistence

The `S3SessionManager` stores conversation state in an S3 bucket.
It uses the same directory structure as `FileSessionManager`, but with S3 keys instead of filesystem paths.

## S3 Key Structure

```
s3://<bucket>/<prefix>/
└── session_<session_id>/
    ├── session.json
    └── agents/
        └── agent_<agent_id>/
            ├── agent.json
            └── messages/
                ├── message_0.json
                └── message_1.json
```

In [ ]:
%pip install -q --upgrade strands-agents boto3

## Configuration

⚠️ **Replace `ACCOUNT_ID` below with your AWS account ID.** The bucket name must be globally unique.

In [ ]:
import boto3

# Replace with your AWS account ID
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REGION = boto3.session.Session().region_name or "us-east-1"

S3_BUCKET = f"strands-sessions-tutorial-{ACCOUNT_ID}"
S3_PREFIX = "tutorial-sessions"
SESSION_ID = "s3-demo-session"

## Create the S3 bucket

In [ ]:
s3 = boto3.client("s3")

try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=S3_BUCKET)
    else:
        s3.create_bucket(
            Bucket=S3_BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Bucket '{S3_BUCKET}' created.")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket '{S3_BUCKET}' already exists.")

## Create an agent with S3SessionManager

In [ ]:
from strands import Agent
from strands.session.s3_session_manager import S3SessionManager

session_manager = S3SessionManager(
    session_id=SESSION_ID,
    bucket=S3_BUCKET,
    prefix=S3_PREFIX,
)

agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=session_manager,
)

response = agent("I'm working on a machine learning pipeline that processes satellite imagery.")
print(response)

In [ ]:
response = agent("The pipeline uses SageMaker for training and Lambda for inference.")
print(response)

## Inspect persisted data in S3

In [ ]:
import json

# List all objects in the session prefix
response_s3 = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=f"{S3_PREFIX}/session_{SESSION_ID}/",
)

print("Objects in S3:")
for obj in response_s3.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

In [ ]:
# Read the session metadata
session_obj = s3.get_object(
    Bucket=S3_BUCKET,
    Key=f"{S3_PREFIX}/session_{SESSION_ID}/session.json",
)
session_data = json.loads(session_obj["Body"].read())
print(json.dumps(session_data, indent=2))

## Restore the session

A new agent with the same session ID picks up where we left off.

In [ ]:
restored_session_manager = S3SessionManager(
    session_id=SESSION_ID,
    bucket=S3_BUCKET,
    prefix=S3_PREFIX,
)

restored_agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=restored_session_manager,
)

response = restored_agent("What services does my pipeline use?")
print(response)

## Cleanup

Delete the session objects and the S3 bucket.

In [ ]:
# Delete all session objects
response_s3 = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=f"{S3_PREFIX}/session_{SESSION_ID}/",
)

for obj in response_s3.get("Contents", []):
    s3.delete_object(Bucket=S3_BUCKET, Key=obj["Key"])

# Delete the bucket
s3.delete_bucket(Bucket=S3_BUCKET)
print(f"Bucket '{S3_BUCKET}' and all session objects deleted.")